# Day 5 / 6: Earnings Call Embeddings

Turn raw text into numbers. Measure which companies talk about similar things.

**Model:** `paraphrase-MiniLM-L3-v2` -- small, fast, no API key needed  
**Exercises:** 6 guided fill-in cells + 1 integration cell  
**Finance context:** Earnings calls as text data for portfolio similarity analysis


In [ ]:
# Cell 0: Setup -- run this first (~20 s on first run)
!pip install -q sentence-transformers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer

np.random.seed(42)

model = SentenceTransformer('paraphrase-MiniLM-L3-v2')
print(f'Ready. Embedding dimension: {model.get_sentence_embedding_dimension()}')


In [ ]:
# Six synthetic earnings call snippets -- read these before starting the exercises
snippets = [
    # 0: TechCore (technology)
    ('TechCore delivered record cloud revenue this quarter, growing 34 percent year-over-year. '
     'Our AI inference platform saw adoption across 12 new enterprise clients. '
     'Capital expenditure for data center expansion reached 2.1 billion dollars. '
     'We guide for continued double-digit growth as enterprises accelerate digital transformation. '
     'Operating margins improved 200 basis points driven by software mix shift.'),

    # 1: ChipMaker (technology)
    ('ChipMaker reported strong demand for high-bandwidth memory in AI accelerator markets. '
     'Our next-generation GPU architecture enters mass production next quarter. '
     'Revenue from the data center segment grew 58 percent versus prior year. '
     'We are expanding fab capacity with a 4 billion dollar investment. '
     'Supply constraints in advanced packaging remain our primary near-term risk.'),

    # 2: FirstBank (banking)
    ('FirstBank reported net interest income of 3.4 billion, up 8 percent on higher deposit spreads. '
     'Credit quality remains solid with non-performing loans at 0.7 percent of total portfolio. '
     'We grew our commercial lending book by 12 percent while maintaining disciplined underwriting. '
     'Tier 1 capital ratio stands at 13.2 percent, well above regulatory minimums. '
     'Provision for credit losses declined as charge-offs trended favorably.'),

    # 3: RegionalBank (banking)
    ('RegionalBank navigated a challenging rate environment with net interest margin compression. '
     'We added 42,000 new checking accounts, demonstrating strong retail franchise strength. '
     'Mortgage origination volume fell 23 percent as refinancing activity slowed. '
     'Cost efficiency ratio improved to 58 percent following branch optimization. '
     'We remain well-capitalized with CET1 ratio of 12.8 percent.'),

    # 4: OilGiant (energy)
    ('OilGiant achieved production of 1.2 million barrels per day, exceeding guidance by 4 percent. '
     'Realized oil prices averaged 82 dollars per barrel, supporting strong free cash flow. '
     'We returned 2.8 billion dollars to shareholders through buybacks and dividends. '
     'Exploration success in the Gulf adds 200 million barrels to proven reserves. '
     'Refining margins benefited from tight distillate markets globally.'),

    # 5: SolarPower (energy)
    ('SolarPower commissioned 3.2 gigawatts of new solar capacity across seven markets. '
     'Revenue grew 41 percent driven by long-term power purchase agreements. '
     'Our battery storage pipeline reached 18 gigawatt-hours, enabling round-the-clock delivery. '
     'Government incentives contributed 340 million dollars to project economics. '
     'We are on track to reach 20 gigawatts of total installed capacity by year-end.'),
]

companies = ['TechCore', 'ChipMaker', 'FirstBank', 'RegionalBank', 'OilGiant', 'SolarPower']
print('Loaded', len(snippets), 'earnings call snippets for:', companies)


## What Embeddings Are

**Similar meaning = nearby vectors.** The model maps each sentence to a point in
a 384-dimensional space. Sentences that discuss the same topic land close together;
unrelated sentences land far apart.

You cannot plot 384 dimensions, but the geometry still works:
- Distance between vectors = semantic distance between meanings
- Cosine similarity (angle between vectors) = a score from 0 (unrelated) to 1 (near-identical)

> **Finance analogy:** Think of an earnings call transcript as a portfolio of topics.
> Two banks discussing NIM, CET1, and NPL ratios will land near each other.
> A tech company discussing GPU margins will land far from both banks.


In [ ]:
# [FULL]
# INSTRUCTOR DEMO: encode 3 short phrases and compare similarity

demo_phrases = [
    'The bank reported strong quarterly profits',     # finance
    'The lender achieved record-high earnings',       # finance (similar meaning)
    'The solar farm expanded its energy capacity',   # energy (different topic)
]

demo_vecs = model.encode(demo_phrases)
demo_sim  = cosine_similarity(demo_vecs)

print('Phrase pairs and their cosine similarity:')
pairs = [(0,1,'bank vs lender (same topic)'), (0,2,'bank vs solar (different topic)')]
for i, j, label in pairs:
    print(f'  {label}: {demo_sim[i,j]:.3f}')
print()
print('Similar-meaning phrases score much higher than unrelated ones.')
print('This pattern holds even though the exact words differ.')


## CHECKPOINT: Predict Similarity Scores

**Before running Cell 1, discuss with your neighbour:**

Look at the 6 earnings snippets above. TechCore and ChipMaker both discuss AI chips
and data centers. FirstBank and RegionalBank both discuss NIM, CET1, and credit quality.

**Predict:** Which pair of companies will have the highest cosine similarity score?

- TechCore vs ChipMaker (both tech)?
- FirstBank vs RegionalBank (both banking)?

Your prediction: ________

*(Run Cells 1-3 to check.)*


## Exercise 1: Encode All Snippets

The model converts each snippet into a vector of 384 numbers.  
Semantically similar sentences produce nearby vectors.


In [ ]:
# Cell 1: Encode snippets

# FILL IN: use model.encode() to convert all snippets into embedding vectors
# embeddings = ______    # expected shape: (6, 384)

# Answer:
embeddings = model.encode(snippets)

print(f'Shape: {embeddings.shape}')   # (6, 384)
print(f'Each snippet is now a vector of {embeddings.shape[1]} numbers')


## Exercise 2: Cosine Similarity Matrix

Cosine similarity = 1 means identical direction; 0 means unrelated.  
We compute all 6 x 6 pairwise scores at once.


In [ ]:
# Cell 2: Cosine similarity matrix

# FILL IN: compute pairwise cosine similarity between all embeddings
# sim_matrix = ______    # expected shape: (6, 6)

# Answer:
sim_matrix = cosine_similarity(embeddings)

df_sim = pd.DataFrame(sim_matrix, index=companies, columns=companies)
print(df_sim.round(3))


In [ ]:
# [FULL]
# WRONG WAY: using dot product instead of cosine_similarity
# Dot product conflates similarity with vector magnitude

a = embeddings[0]   # TechCore
b = embeddings[1]   # ChipMaker  (tech -- should be similar)
c = embeddings[2]   # FirstBank  (bank -- should be less similar)

dot_ab = np.dot(a, b)
dot_ac = np.dot(a, c)
cos_ab = cosine_similarity(a.reshape(1,-1), b.reshape(1,-1))[0, 0]
cos_ac = cosine_similarity(a.reshape(1,-1), c.reshape(1,-1))[0, 0]

print('Dot product (WRONG WAY):')
print(f'  TechCore vs ChipMaker  = {dot_ab:.1f}')
print(f'  TechCore vs FirstBank  = {dot_ac:.1f}')
print()
print('Cosine similarity (correct):')
print(f'  TechCore vs ChipMaker  = {cos_ab:.3f}')
print(f'  TechCore vs FirstBank  = {cos_ac:.3f}')
print()
print('Problem: dot product values are large and hard to compare across pairs.')
print('They grow with vector magnitude, not just angle.')


## Why Cosine Similarity, Not Dot Product

The dot product of two vectors equals `|a| * |b| * cos(theta)` -- it mixes
the angle (what we want) with the magnitudes (not relevant for semantic similarity).

**Cosine similarity** divides by both magnitudes, leaving only the angle:
`cos(theta) = dot(a, b) / (|a| * |b|)`. Result is always in [-1, 1].

Sentence-transformer embeddings are L2-normalised (all vectors have magnitude 1),
so `dot(a, b) == cosine_similarity(a, b)` for this model. But that is a special
property of this model -- do not rely on it in general. Use `cosine_similarity` explicitly.


## CHECKPOINT: Predict the Most Similar Pair

**Before running Cell 3, discuss with your neighbour:**

From the similarity matrix (Cell 2), which off-diagonal entry do you expect to be largest?

- TechCore vs ChipMaker (cloud + AI language)?
- FirstBank vs RegionalBank (banking metrics language)?
- OilGiant vs SolarPower (both energy, but very different vocabulary)?

Your prediction: ________

*(Run Cell 3 to check.)*


## Exercise 3: Most Similar Pair

The diagonal is always 1.0 (a snippet compared to itself).  
Ignore it and find the largest off-diagonal score.


In [ ]:
# Cell 3: Most similar pair

sim_no_diag = sim_matrix.copy()
np.fill_diagonal(sim_no_diag, 0)

# FILL IN: find the flat index of the maximum value in sim_no_diag
# idx_flat = ______
# i, j = np.unravel_index(idx_flat, sim_no_diag.shape)

# Answer:
idx_flat = np.argmax(sim_no_diag)
i, j = np.unravel_index(idx_flat, sim_no_diag.shape)

print(f'Most similar pair: {companies[i]} and {companies[j]}  (score {sim_matrix[i, j]:.3f})')


## Exercise 4: Rank Snippets by Similarity to a Query

Use snippet 0 (TechCore) as the query.  
Which of the other five companies is most similar to TechCore?


In [ ]:
# Cell 4: Rank by similarity to TechCore

query_idx  = 0
query_name = companies[query_idx]

# FILL IN: extract the similarity scores for the query snippet
# scores = ______    # hint: use sim_matrix[query_idx]

# Answer:
scores = sim_matrix[query_idx]

ranked = np.argsort(scores)[::-1]
print(f'Ranked by similarity to {query_name}:')
for rank, idx in enumerate(ranked):
    print(f'  {rank + 1}. {companies[idx]:15s}  {scores[idx]:.3f}')


## Exercise 5: Interpret a Similarity Score

Read the three scores printed below.  
Then complete the interpretation sentences in the comments.


In [ ]:
# Cell 5: Interpret similarity scores

print(f'TechCore vs ChipMaker  (both tech):     {sim_matrix[0, 1]:.3f}')
print(f'TechCore vs FirstBank  (tech vs bank):  {sim_matrix[0, 2]:.3f}')
print(f'TechCore vs OilGiant   (tech vs energy):{sim_matrix[0, 4]:.3f}')

# FILL IN: complete the three sentences below
# A score close to 1.0 means:  ______
# A score close to 0.0 means:  ______
# Same-sector pairs score higher because: ______

# Answer:
# A score close to 1.0 means: the two snippets discuss very similar topics.
# A score close to 0.0 means: the snippets discuss essentially unrelated subjects.
# Same-sector pairs score higher because they share industry vocabulary and
# report identical metrics (margins, capex, revenue growth rates).


## Exercise 6: Which Companies Discuss Similar Topics?

Visualize the full matrix as a heatmap.  
Look for bright blocks -- those are clusters of similar companies.


In [ ]:
# Cell 6: Heatmap and sector identification

NAVY = '#1e3a5f'

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(6))
ax.set_yticks(range(6))
ax.set_xticklabels(companies, rotation=40, ha='right', fontsize=10)
ax.set_yticklabels(companies, fontsize=10)
for ii in range(6):
    for jj in range(6):
        ax.text(jj, ii, f'{sim_matrix[ii, jj]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, label='Cosine Similarity')
ax.set_title('Earnings Call Similarity Matrix', fontsize=12, color=NAVY)
plt.tight_layout()
plt.show()

# FILL IN: which pair has the highest within-sector similarity?
# top_pair = '______'    # e.g. 'FirstBank and RegionalBank'
# reason   = '______'    # one sentence explaining why

# Answer:
# The banking pair (FirstBank / RegionalBank) typically shows the highest within-sector
# similarity because both report identical regulatory metrics: NIM, CET1, NPL ratios.


---
## Integration Cell -- Day 6, Phase 2 (60 min)

Combine three techniques from the course:

1. **Logistic Regression** on tabular loan data (Days 3-4)
2. **KMeans** on text embedding vectors (Day 5 morning)
3. **Cosine similarity** to a risk vocabulary query (just learned)

The same distance operation works on both text and numbers -- that is the key insight.


In [ ]:
# Cell 7: Integration -- Loan classification + Earnings clustering

np.random.seed(42)

# 1. Synthetic loan dataset (200 applicants, 4 features)
n            = 200
income       = np.random.uniform(20_000, 120_000, n)
debt_ratio   = np.random.uniform(0.1, 0.9, n)
credit_score = np.random.uniform(300, 850, n)
loan_amount  = np.random.uniform(5_000, 50_000, n)
X = np.column_stack([income, debt_ratio, credit_score, loan_amount])

# Label: approved when credit_score > 600 AND debt_ratio < 0.5 AND income > 50k, plus 10% noise
base  = ((credit_score > 600) & (debt_ratio < 0.5) & (income > 50_000)).astype(int)
noise = np.random.binomial(1, 0.10, n)
y     = np.where(noise, 1 - base, base)

# 2. Train Logistic Regression
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)
clf    = LogisticRegression(random_state=42, max_iter=200)
clf.fit(X_sc, y)
print(f'Classifier training accuracy: {clf.score(X_sc, y):.3f}')

# 3. Cluster the 6 earnings snippets using their embeddings
kmeans         = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)
print()
print('Earnings snippet clusters:')
for company, lbl in zip(companies, cluster_labels):
    print(f'  {company:15s} --> Cluster {lbl}')

# 4. Which cluster is most similar to a credit-risk vocabulary query?
risk_query     = 'loan default credit risk non-performing provision charge-off'
risk_vec       = model.encode([risk_query])
centroid_sims  = cosine_similarity(risk_vec, kmeans.cluster_centers_)[0]
risk_cluster   = int(np.argmax(centroid_sims))
risk_companies = [c for c, l in zip(companies, cluster_labels) if l == risk_cluster]
print()
print(f'Cluster closest to credit-risk vocabulary: Cluster {risk_cluster}')
print(f'Companies in that cluster: {risk_companies}')

# 5. Predict for a sample applicant
sample    = np.array([[80_000, 0.30, 720, 20_000]])
sample_sc = scaler.transform(sample)
pred      = clf.predict(sample_sc)[0]
outcome   = 'APPROVED' if pred == 1 else 'DENIED'
print()
print(f'Sample (income=80k, debt=0.30, score=720, loan=20k) --> {outcome}')


## Embeddings as a Text Factor Model

In quantitative finance, a **factor model** decomposes asset returns into exposures
to underlying factors (market, value, momentum). Each stock gets a vector of factor
loadings that determines how it moves.

Sentence embeddings do the same thing for text. Each earning call snippet gets a
384-dimensional vector of 'topic loadings'. Companies with similar topic profiles
(both tech, both banking) get similar loading vectors -- and land near each other.

This connection runs deep:
- **Factor analysis** on returns matrix (numeric) finds latent risk factors
- **Embedding model** on text corpus (language) finds latent topic factors
- **Cosine similarity** on embedding vectors = correlation of topic exposures

The integration cell (Cell 7) used cosine similarity to identify which cluster of
earnings calls is semantically closest to a credit-risk vocabulary -- exactly the
same operation as computing factor exposure in a returns-based model.

> **Key insight:** Distance is distance. Whether the vectors come from text or returns,
> the geometry of similarity is the same operation.


## Synthesis: Fill in the Blanks

Discuss with your group and complete these sentences:

**Company ________ is in Cluster ___ because ________________________________.**

**The classifier predicts ___ (APPROVED / DENIED) for the sample applicant because ________________________________.**

**Embeddings group companies by _______________, while the classifier separates applicants by _______________.**

---
*Key insight:* Cosine similarity is the same operation whether vectors represent text or tabular data. Distance is distance.
